# FairPay: Salary Prediction - Exploratory Data Analysis

This notebook performs exploratory data analysis on the synthetic salary dataset.
We examine distributions, missing values, correlations, and relationships between features and salary.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load data
data_path = Path('../data/salary_data.csv')
df = pd.read_csv(data_path)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Descriptive Statistics ===')
print(df.describe(include='all').T)

## Missing Values Analysis

In [ ]:
# Visualize missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
missing_df['Missing %'].plot(kind='bar', ax=ax, color='coral')
ax.set_title('Missing Values by Column')
ax.set_ylabel('Missing Percentage')
ax.set_xlabel('Column')
plt.tight_layout()
plt.show()

print(missing_df)

## Target Variable Distribution (Salary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['salary'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_title('Salary Distribution')
axes[0].set_xlabel('Salary ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['salary'].mean(), color='red', linestyle='--', label=f'Mean: ${df["salary"].mean():,.0f}')
axes[0].axvline(df['salary'].median(), color='green', linestyle='--', label=f'Median: ${df["salary"].median():,.0f}')
axes[0].legend()

# Box plot
axes[1].boxplot(df['salary'], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red'))
axes[1].set_title('Salary Box Plot')
axes[1].set_ylabel('Salary ($)')

plt.tight_layout()
plt.show()

print(f'Skewness: {df["salary"].skew():.3f}')
print(f'Kurtosis: {df["salary"].kurtosis():.3f}')

## Numerical Features Distributions

In [ ]:
num_cols = ['years_experience', 'skills_count', 'previous_salary', 'interview_score', 'age', 'salary']
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_title(f'{col.replace("_", " ").title()} Distribution')
    axes[i].set_xlabel(col.replace('_', ' ').title())
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Categorical Features Distributions

In [ ]:
cat_cols = ['education_level', 'job_role', 'location', 'company_size', 'gender']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[i].bar(range(len(counts)), counts.values, color='mediumseagreen', edgecolor='black', alpha=0.7)
    axes[i].set_xticks(range(len(counts)))
    axes[i].set_xticklabels(counts.index, rotation=45, ha='right')
    axes[i].set_title(f'{col.replace("_", " ").title()} Distribution')
    axes[i].set_ylabel('Count')

# Remove empty subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

## Salary by Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # Order by median salary
    order = df.groupby(col)['salary'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='salary', order=order, ax=axes[i], palette='Set2')
    axes[i].set_title(f'Salary by {col.replace("_", " ").title()}')
    axes[i].set_xlabel(col.replace('_', ' ').title())
    axes[i].set_ylabel('Salary ($)')
    axes[i].tick_params(axis='x', rotation=45)

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

## Salary by Numerical Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(['years_experience', 'skills_count', 'previous_salary', 'interview_score', 'age']):
    axes[i].scatter(df[col], df['salary'], alpha=0.3, s=10, color='darkorange')
    axes[i].set_title(f'Salary vs {col.replace("_", " ").title()}')
    axes[i].set_xlabel(col.replace('_', ' ').title())
    axes[i].set_ylabel('Salary ($)')
    # Add trend line
    z = np.polyfit(df[col].dropna(), df.loc[df[col].notna(), 'salary'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_range, p(x_range), 'r--', linewidth=2)

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
num_df = df[num_cols].copy()
corr_matrix = num_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix - Numerical Features')
plt.tight_layout()
plt.show()

print('Correlation with salary:')
print(corr_matrix['salary'].sort_values(ascending=False))

## Salary by Experience and Role (Combined View)

In [ ]:
# Salary by experience bins and role
df['exp_bin'] = pd.cut(df['years_experience'], bins=[0, 2, 5, 10, 15, 30], 
                        labels=['0-2', '2-5', '5-10', '10-15', '15+'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot
sns.boxplot(data=df, x='exp_bin', y='salary', hue='job_role', ax=axes[0], palette='tab20')
axes[0].set_title('Salary by Experience Bin and Job Role')
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Salary ($)')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

# Median salary heatmap
pivot = df.pivot_table(values='salary', index='job_role', columns='exp_bin', aggfunc='median')
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[1], cbar_kws={'label': 'Median Salary'})
axes[1].set_title('Median Salary by Role and Experience')
axes[1].set_xlabel('Years of Experience')
axes[1].set_ylabel('Job Role')

plt.tight_layout()
plt.show()

## Fairness Check: Salary by Protected Attributes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Salary by gender
sns.boxplot(data=df, x='gender', y='salary', ax=axes[0], palette='Set2')
axes[0].set_title('Salary by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Salary ($)')

# Salary by age groups
df['age_group'] = pd.cut(df['age'], bins=[20, 30, 40, 50, 65], 
                          labels=['20-30', '30-40', '40-50', '50+'])
sns.boxplot(data=df, x='age_group', y='salary', ax=axes[1], palette='Set2')
axes[1].set_title('Salary by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Salary ($)')

plt.tight_layout()
plt.show()

print('Mean salary by gender:')
print(df.groupby('gender')['salary'].mean().round(0))

print('\nMean salary by age group:')
print(df.groupby('age_group')['salary'].mean().round(0))

## Outlier Detection

In [ ]:
# IQR method for outlier detection
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['salary'] < lower_bound) | (df['salary'] > upper_bound)]
print(f'Number of outliers (IQR method): {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')
print(f'Lower bound: ${lower_bound:,.0f}')
print(f'Upper bound: ${upper_bound:,.0f}')
print(f'Outlier salary range: ${outliers["salary"].min():,.0f} - ${outliers["salary"].max():,.0f}')

# Show outlier roles
print('\nOutlier job roles:')
print(outliers['job_role'].value_counts())

## Key Insights Summary

In [ ]:
insights = [
    "1. **Salary Distribution**: Right-skewed with mean ($205K) > median ($196K), typical for salary data.",
    "2. **Missing Values**: ~2% missing in previous_salary, interview_score, skills_count - will need imputation.",
    "3. **Top Correlations with Salary": previous_salary (0.72), years_experience (0.68), interview_score (0.45), skills_count (0.38).",
    "4. **Job Role Impact**: Engineering Manager and ML Engineer have highest median salaries; Data Analyst lowest.",
    "5. **Location Impact": San Francisco and New York have highest salaries; Remote and Chicago lowest.",
    "6. **Experience**: Strong positive correlation with salary, but plateaus after ~15 years.",
    "7. **Protected Attributes": Gender and age show salary differences - MUST audit for fairness in modeling.",
    "8. **Outliers**: ~3% outliers in salary (mostly high-salary Engineering Managers in SF/NY) - consider robust modeling."
]

for insight in insights:
    print(insight)